# 01 · Auditoría de datos y construcción de variables

**Objetivo.** Verificar que la base sea íntegra y coherente, y construir las variables analíticas que se usarán en el resto del análisis. El resultado es una base analítica única (`data/processed/caso05_analitico.csv`).

**Justificación.** Todo efecto estimado depende de la calidad de los datos. Un identificador duplicado, una variable binaria mal codificada o un faltante mal tratado pueden sesgar el resultado más que cualquier decisión econométrica. Además, construir las variables en un único lugar asegura que todos los notebooks usen exactamente las mismas definiciones.

**Fundamento metodológico.** Las verificaciones siguen el orden habitual de una auditoría de datos, de lo más básico a lo más fino: (1) estructura y tipos, (2) unidad de observación, (3) codificación, (4) rangos plausibles, (5) consistencia lógica entre variables, (6) faltantes y su mecanismo. La construcción de variables se limita a transformaciones **definidas antes de estimar** y documentadas en un diccionario.

### Preparación del entorno

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.float_format", "{:,.3f}".format)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

# Rutas relativas a la raíz del repositorio
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw" / "caso05_cuidado_infantil.dta"
PROCESSED = ROOT / "data" / "processed" / "caso05_analitico.csv"

# Roles de las variables (definidos en 00_planteamiento_teoria_cambio)
ID, CENTER = "hogar_id", "centro_id"
Z = "oferta_sorteo"            # asignación aleatoria (instrumento)
D = "uso_horario_ampliado"     # tratamiento recibido (endógeno)
BASELINE = [
    "edad_madre", "numero_hijos", "edad_menor", "educ_madre", "madre_sola",
    "traslado_min", "red_cuidado", "empleo_pre", "horas_pre", "ingreso_pre",
]
OUTCOMES = ["empleo_post", "horas_post", "ingreso_post_c", "bienestar_post"]
ALPHA, SEED = 0.05, 2026

# Estilo gráfico: oferta = azul, sin oferta = naranja; grilla recesiva
COLORS = {0: "#eb6834", 1: "#2a78d6"}
LABELS = {0: "Sin oferta (Z=0)", 1: "Con oferta (Z=1)"}
NEUTRAL = "#8a8984"
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "grid.color": "#e6e6e3", "figure.dpi": 110, "axes.titleweight": "bold",
})

In [2]:
import pyreadstat

def read_stata(filepath, columns=None):
    # Lee un .dta y devuelve (datos, metadatos). Valida existencia y extensión.
    path = Path(filepath)
    if not path.is_file():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")
    if path.suffix.lower() != ".dta":
        raise ValueError("Se requiere un archivo .dta")
    return pyreadstat.read_dta(str(path), usecols=columns, apply_value_formats=False)

df_raw, meta = read_stata(RAW)
df = df_raw.copy()
BINARY = [Z, D, "madre_sola", "red_cuidado", "empleo_pre", "empleo_post"]
print(f"Dimensiones: {df.shape}")

Dimensiones: (1500, 18)


## 1.1 Estructura y tipos

Se contrastan las dimensiones, los tipos y las etiquetas con el diccionario de datos del curso (no publicado): 1 500 observaciones y 18 variables.

In [3]:
estructura = pd.DataFrame({
    "etiqueta": [meta.column_names_to_labels.get(c, "") for c in df.columns],
    "tipo": df.dtypes.astype(str).values,
    "n_unicos": df.nunique().values,
    "faltantes": df.isna().sum().values,
}, index=df.columns)
display(estructura)

,etiqueta,tipo,n_unicos,faltantes
hogar_id,Identificador del hogar postulante,int64,1500,0
centro_id,Centro de cuidado solicitado,int64,5,0
oferta_sorteo,Seleccionada por sorteo para vacante,int64,2,0
uso_horario_ampliado,Uso efectivo del horario ampliado,int64,2,0
edad_madre,Edad de la madre,int64,28,0
numero_hijos,Numero de hijos,int64,4,0
edad_menor,Edad del hijo menor,int64,5,0
educ_madre,Anos de educacion de la madre,int64,16,0
madre_sola,Hogar monoparental con madre,int64,2,0
traslado_min,Minutos de traslado al empleo,float64,1472,0


**Interpretación.** Las 18 variables del diccionario están presentes, con sus etiquetas y 1 500 filas. Todas son numéricas: enteras (`int64`) las de conteo o sí/no, y con decimales (`float64`) traslado, horas, ingreso y bienestar. Aun así, se valida la codificación de las binarias en 1.3. La **única variable con faltantes es `ingreso_post`**, que se analiza en 1.6.

## 1.2 Unidad de observación

Según el diccionario, cada fila es una madre u hogar postulante: esa es la **unidad de observación** (el "quién" de cada fila). Se verifica que no haya identificadores repetidos ni registros duplicados con distinto identificador (por ejemplo, una misma postulante registrada dos veces).

In [4]:
sin_id = [c for c in df.columns if c != ID]
diagnostico_id = pd.Series({
    "filas": len(df),
    "ids únicos": df[ID].nunique(),
    "ids faltantes": df[ID].isna().sum(),
    "filas idénticas salvo el id": df.duplicated(subset=sin_id).sum(),
})
display(diagnostico_id.to_frame("valor"))

,valor
filas,1500
ids únicos,1500
ids faltantes,0
filas idénticas salvo el id,0


**Interpretación.** Cada fila tiene un identificador distinto y no hay registros idénticos en las demás variables. Se confirma la unidad de observación (**1 fila = 1 postulante**). Esto importa porque cada fila se tratará como una unidad independiente en la aleatorización y en la inferencia.

## 1.3 Codificación de variables binarias

In [5]:
codificacion = pd.DataFrame({
    v: {"valores": sorted(df[v].dropna().unique().tolist()),
        "etiquetas_dta": meta.variable_value_labels.get(v, {}),
        "media (=% sí)": df[v].mean()}
    for v in BINARY
}).T
display(codificacion)
assert all(set(df[v].dropna().unique()) <= {0, 1} for v in BINARY), "Binaria con valores fuera de {0,1}"

,valores,etiquetas_dta,media (=% sí)
oferta_sorteo,"[0, 1]","{0: 'No', 1: 'Si'}",0.499
uso_horario_ampliado,"[0, 1]","{0: 'No', 1: 'Si'}",0.514
madre_sola,"[0, 1]","{0: 'No', 1: 'Si'}",0.304
red_cuidado,"[0, 1]","{0: 'No', 1: 'Si'}",0.438
empleo_pre,"[0, 1]","{0: 'No', 1: 'Si'}",0.469
empleo_post,"[0, 1]","{0: 'No', 1: 'Si'}",0.618


**Interpretación.** Las seis variables binarias toman solo los valores {0, 1}, y las etiquetas del archivo confirman que 1 = "Sí". Sus medias son directamente proporciones. Por ejemplo, el 50 % de las postulantes fue sorteada y el 51 % usó el horario ampliado. Que el uso sea casi igual a la oferta *en el agregado* no significa cumplimiento perfecto: se examina en 02.

## 1.4 Rangos plausibles

Se definen rangos admisibles **a priori**, según el significado de cada variable, y se cuentan los valores que caen fuera.

In [6]:
reglas = {
    "edad_madre": (14, 60), "numero_hijos": (1, 15), "edad_menor": (0, 6),
    "educ_madre": (0, 22), "traslado_min": (0, 240),
    "horas_pre": (0, 98), "horas_post": (0, 98),
    "ingreso_pre": (0, None), "ingreso_post": (0, None), "bienestar_post": (0, 100),
}
rangos = []
for v, (lo, hi) in reglas.items():
    s = df[v].dropna()
    fuera = (s < lo).sum() + ((s > hi).sum() if hi is not None else 0)
    rangos.append({"variable": v, "mín": s.min(), "p50": s.median(), "máx": s.max(),
                   "regla": f"[{lo}, {hi if hi is not None else '∞'}]", "fuera_de_rango": fuera})
display(pd.DataFrame(rangos).set_index("variable"))

,mín,p50,máx,regla,fuera_de_rango
variable,,,,,
edad_madre,18.000,29.000,45.000,"[14, 60]",0
numero_hijos,1.000,1.000,4.000,"[1, 15]",0
edad_menor,1.000,3.000,5.000,"[0, 6]",0
educ_madre,3.000,12.000,18.000,"[0, 22]",0
traslado_min,5.000,42.117,108.448,"[0, 240]",0
horas_pre,0.000,0.000,50.000,"[0, 98]",0
horas_post,0.000,27.104,55.000,"[0, 98]",0
ingreso_pre,0.000,0.000,"2,571.158","[0, ∞]",0
ingreso_post,0.000,"1,036.985","2,763.147","[0, ∞]",0


**Unidades.** El diccionario no indica la moneda del ingreso; en todos los notebooks se expresa en **unidades monetarias (u.m.)** mensuales.

**Interpretación.** Ninguna observación está fuera de los rangos admisibles. Edades de 18 a 45 años, entre 1 y 4 hijos, hijo menor de 1 a 5 años (edad preescolar, coherente con un servicio de cuidado infantil), escolaridad entre 3 y 18 años, ingresos no negativos y bienestar dentro de 0-100. No hace falta recortar ni corregir valores extremos en esta etapa. La sensibilidad a valores altos de ingreso se evalúa en 05.

## 1.5 Consistencia lógica entre empleo, horas e ingreso

**Regla:** si `empleo = 0`, entonces `horas = 0` e `ingreso = 0`. Si `empleo = 1`, entonces `horas > 0` e `ingreso > 0` (cuando se observa). También se verifica que la madre tuviera al menos 12 años al nacer su hijo menor.

In [7]:
consistencia = {}
for t in ["pre", "post"]:
    e, h, y = df[f"empleo_{t}"], df[f"horas_{t}"], df[f"ingreso_{t}"]
    consistencia[t] = {
        "no empleada con horas > 0": ((e == 0) & (h > 0)).sum(),
        "no empleada con ingreso > 0": ((e == 0) & (y > 0)).sum(),
        "empleada con horas = 0": ((e == 1) & (h == 0)).sum(),
        "empleada con ingreso = 0": ((e == 1) & (y == 0)).sum(),
        "no empleada con ingreso faltante": ((e == 0) & y.isna()).sum(),
        "empleada con ingreso faltante": ((e == 1) & y.isna()).sum(),
    }
display(pd.DataFrame(consistencia))
print("Madres con edad al nacer el hijo menor < 12:",
      ((df["edad_madre"] - df["edad_menor"]) < 12).sum())

,pre,post
no empleada con horas > 0,0,0
no empleada con ingreso > 0,0,0
empleada con horas = 0,0,0
empleada con ingreso = 0,0,0
no empleada con ingreso faltante,0,19
empleada con ingreso faltante,0,39


Madres con edad al nacer el hijo menor < 12: 0


**Interpretación**

- No hay contradicciones: en ambos períodos, la condición de empleo coincide con horas e ingreso positivos.
- **Hallazgo clave para los faltantes:** de los 58 ingresos posteriores faltantes, **19 son de madres no empleadas** en el seguimiento. Para ellas el ingreso laboral es **0 por definición**: las 554 no empleadas con ingreso observado tienen exactamente 0. Esos 19 casos son faltantes **estructurales** y se pueden completar con una regla lógica, sin un modelo. Solo **39 faltantes** (2.6 %) son pérdidas reales de información entre empleadas.

## 1.6 Valores faltantes

**Fundamento metodológico.** Los faltantes se clasifican según su mecanismo [1]:

| Tipo | En sencillo | Ejemplo en este caso |
|---|---|---|
| **MCAR** (completamente al azar) | Falta por pura mala suerte, sin relación con nada | Se perdió la hoja de la encuesta |
| **MAR** (al azar dado lo observado) | Falta más en ciertos grupos que se pueden identificar con los datos | Falta más en madres con menos educación, y la educación está en la base |
| **MNAR** (no al azar) | Falta *por* el propio valor que no se ve | Las madres que ganan muy poco prefieren no declarar su ingreso |

En un experimento, lo crítico es si la falta de datos difiere entre grupos de asignación. Eso se llama **atrición diferencial**: si a las sorteadas les falta el dato por razones distintas que a las no sorteadas, los grupos con dato dejan de ser comparables, aunque el sorteo los hiciera comparables al inicio. Aquí solo se describe el patrón. La prueba formal está en 02 y la robustez en 05.

In [8]:
falta = df["ingreso_post"].isna()
patron = (
    df.assign(falta=falta)
    .groupby([Z, "empleo_post"])["falta"]
    .agg(n="size", faltantes="sum", tasa="mean")
)
display(patron)

n  faltantes  tasa
oferta_sorteo empleo_post                      
0             0            323         12 0.037
              1            428         20 0.047
1             0            250          7 0.028
              1            499         19 0.038

**Interpretación.** Las tasas de faltantes son bajas (2.8 % a 4.7 %) y alrededor de 1 punto porcentual más altas en el grupo sin oferta, tanto entre empleadas como entre no empleadas. Una diferencia de ese tamaño no indica, por sí sola, atrición diferencial. Se prueba formalmente en 02.7, después de la imputación lógica.

## 1.7 Construcción de variables

**Objetivo.** Generar las variables que exigen las preguntas causales (definidas en 00), con reglas explícitas.

| Variable | Definición | Uso |
|---|---|---|
| `ingreso_post_c` | `ingreso_post`, con 0 si `empleo_post = 0` (imputación lógica) | Resultado de ingreso principal |
| `ingreso_imput_logica` | 1 si se completó por la regla anterior | Trazabilidad |
| `R_ingreso` | 1 si `ingreso_post_c` está observado | Análisis de faltantes (02, 05) |
| `d_empleo`, `d_horas`, `d_ingreso` | Cambio entre la línea base y el seguimiento | Descriptivo; robustez (resultado en diferencias) |
| `sin_red` | 1 − `red_cuidado` | Heterogeneidad (H5) |
| `restriccion_alta` | Madre sola **y** sin red de cuidado | Heterogeneidad (H5) |
| `celda_cumplimiento` | Combinación observada de $Z$ y $D$ | Cumplimiento (02, 04) |
| `indice_laboral` | Promedio de los puntajes $z$ disponibles de empleo, horas e ingreso, estandarizados con la media y la desviación estándar del grupo sin oferta; variante de [2] | Resumen único de los resultados laborales; reduce la multiplicidad |

> **En sencillo — puntaje $z$ e índice resumen:** un puntaje $z$ expresa cada valor como "cuántas desviaciones estándar está por encima o por debajo del promedio del grupo sin oferta". Así, empleo (0/1), horas e ingreso (u.m.) quedan en una misma escala y se pueden promediar. Un índice de +0.2 significa que, en conjunto, la madre está 0.2 desviaciones estándar por encima de la típica no sorteada. Resumir tres resultados en uno reduce el riesgo de encontrar un efecto "por casualidad" al probar muchas cosas (multiplicidad, ver 04).

> **En sencillo — imputación:** completar un dato faltante. La **imputación lógica** usa una regla que no admite duda (si no trabaja, su ingreso laboral es 0). La **imputación estadística** adivina el valor con un modelo y siempre tiene incertidumbre.

**¿Por qué la imputación lógica y no una imputación estadística?** Porque para esas 19 madres el valor no es desconocido: el ingreso laboral de quien no trabaja es 0. No completarlo descartaría información y haría que la muestra de ingreso dependiera del empleo (el colisionador señalado en 00).

In [9]:
# Ingreso posterior con imputación lógica (0 si no está empleada)
df["ingreso_imput_logica"] = (df["ingreso_post"].isna() & (df["empleo_post"] == 0)).astype(int)
df["ingreso_post_c"] = np.where(df["empleo_post"] == 0, 0.0, df["ingreso_post"])
df["R_ingreso"] = df["ingreso_post_c"].notna().astype(int)

# Cambios entre la línea base y el seguimiento
for v in ["empleo", "horas"]:
    df[f"d_{v}"] = df[f"{v}_post"] - df[f"{v}_pre"]
df["d_ingreso"] = df["ingreso_post_c"] - df["ingreso_pre"]

# Restricciones de cuidado (heterogeneidad preespecificada)
df["sin_red"] = 1 - df["red_cuidado"]
df["restriccion_alta"] = ((df["madre_sola"] == 1) & (df["sin_red"] == 1)).astype(int)

# Celda observada de cumplimiento
df["celda_cumplimiento"] = df[Z].astype(int).astype(str) + "-" + df[D].astype(int).astype(str)
df["celda_cumplimiento"] = df["celda_cumplimiento"].map({
    "1-1": "Sorteada y usa", "1-0": "Sorteada, no usa",
    "0-1": "No sorteada, usa", "0-0": "No sorteada, no usa",
})

# Índice laboral: puntajes z respecto del grupo sin oferta, promedio de los disponibles
def indice_kling(data, componentes, grupo):
    ctrl = data.loc[data[grupo] == 0, componentes]
    z = (data[componentes] - ctrl.mean()) / ctrl.std()
    return z.mean(axis=1, skipna=True)

df["indice_laboral"] = indice_kling(df, ["empleo_post", "horas_post", "ingreso_post_c"], Z)

print("Faltantes de ingreso: original =", df["ingreso_post"].isna().sum(),
      "| imputación lógica =", df["ingreso_imput_logica"].sum(),
      "| restantes =", df["ingreso_post_c"].isna().sum())

Faltantes de ingreso: original = 58 | imputación lógica = 19 | restantes = 39


In [10]:
# Validaciones de las variables construidas
assert (df.loc[df["empleo_post"] == 0, "ingreso_post_c"] == 0).all()
assert df["ingreso_post_c"].isna().sum() == (df["ingreso_post"].isna() & (df["empleo_post"] == 1)).sum()
assert df["indice_laboral"].notna().all()
nuevas = ["ingreso_post_c", "ingreso_imput_logica", "R_ingreso", "d_empleo", "d_horas",
          "d_ingreso", "sin_red", "restriccion_alta", "indice_laboral"]
display(df[nuevas].describe().T)
display(df["celda_cumplimiento"].value_counts().to_frame("n"))

# El índice se estandariza con el grupo sin oferta: su media ≈ 0 y DE ≈ 1 deben verse en ese grupo, no en el total
indice_grupo = df.groupby(Z)["indice_laboral"].agg(media="mean", de="std").rename(index=LABELS)
indice_grupo.loc["Toda la muestra"] = [df["indice_laboral"].mean(), df["indice_laboral"].std()]
display(indice_grupo)

,count,mean,std,min,25%,50%,75%,max
ingreso_post_c,"1,461.000",876.853,777.256,0.000,0.000,"1,026.906","1,538.917","2,763.147"
ingreso_imput_logica,"1,500.000",0.013,0.112,0.000,0.000,0.000,0.000,1.000
R_ingreso,"1,500.000",0.974,0.159,0.000,1.000,1.000,1.000,1.000
d_empleo,"1,500.000",0.149,0.659,-1.000,0.000,0.000,1.000,1.000
d_horas,"1,500.000",7.125,21.667,-50.000,0.000,6.578,24.513,55.000
d_ingreso,"1,461.000",234.537,927.565,"-2,316.177",-57.667,0.000,938.978,"2,476.608"
sin_red,"1,500.000",0.562,0.496,0.000,0.000,1.000,1.000,1.000
restriccion_alta,"1,500.000",0.167,0.373,0.000,0.000,0.000,0.000,1.000
indice_laboral,"1,500.000",0.126,0.976,-1.074,-1.074,0.566,0.954,1.678


,n
celda_cumplimiento,
Sorteada y usa,619
"No sorteada, no usa",599
"No sorteada, usa",152
"Sorteada, no usa",130


,media,de
oferta_sorteo,,
Sin oferta (Z=0),0.007,0.971
Con oferta (Z=1),0.245,0.968
Toda la muestra,0.126,0.976


**Interpretación**

- Después de la imputación lógica quedan **39 faltantes de ingreso** (2.6 %), todos de madres empleadas.
- `restriccion_alta` identifica a las madres solas sin red de cuidado, el grupo en que la teoría de cambio anticipa el mayor efecto.
- Las cuatro celdas de cumplimiento muestran desde ya que **el cumplimiento es imperfecto en ambos sentidos**: hay sorteadas que no usan el servicio y no sorteadas que sí lo usan. Esto obliga a distinguir ITT y LATE (02, 04).
- **Índice laboral:** en el grupo sin oferta tiene media ≈ 0 (0.007) y desviación estándar ≈ 1 (0.97), porque se estandariza respecto de ese grupo. En el grupo con oferta la media es 0.245. La media de 0.126 que aparece en la tabla descriptiva anterior es la de **toda la muestra**: promedia ambos grupos. La diferencia entre grupos (≈ 0.24) se lee aproximadamente como efecto en desviaciones estándar. Es una estimación descriptiva; la formal está en 04.
- Es una **variante** del índice de Kling, Liebman y Katz [2]: en lugar de imputar la media del grupo al componente faltante (como hacen los autores), se promedian los componentes disponibles. Por eso las 39 madres sin ingreso tienen un índice de dos componentes.

## 1.8 Guardar la base analítica

In [11]:
PROCESSED.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED, index=False)
print(f"Guardado: {PROCESSED.relative_to(ROOT)}  ({df.shape[0]} x {df.shape[1]})")

Guardado: data/processed/caso05_analitico.csv  (1500 x 28)


## 1.9 Síntesis, supuestos y decisiones

| Verificación | Resultado |
|---|---|
| Estructura | 1 500 × 18, coincide con el diccionario |
| Unidad de observación | Única por fila; sin duplicados |
| Codificación | Binarias en {0, 1}; 1 = Sí |
| Rangos y consistencia | Sin valores imposibles ni contradicciones |
| Faltantes | Solo `ingreso_post`: 58 → 39 después de la imputación lógica |

**Supuestos.** La imputación lógica supone que `ingreso_post` mide **solo ingreso laboral**, como indica el diccionario. Si incluyera otras fuentes, completar con 0 sería incorrecto.

**Decisiones para las etapas siguientes**

- El resultado de ingreso es `ingreso_post_c`. La versión original se usa como contraste de robustez en 05.
- Los 39 faltantes restantes se tratan con un análisis de casos completos en la especificación principal, junto con diagnóstico de atrición (02) y cotas (05).

## Referencias

1. Rubin, D. B. (1976). Inference and missing data. *Biometrika*, 63(3), 581-592. https://doi.org/10.1093/biomet/63.3.581
2. Kling, J. R., Liebman, J. B. y Katz, L. F. (2007). Experimental analysis of neighborhood effects. *Econometrica*, 75(1), 83-119. https://doi.org/10.1111/j.1468-0262.2007.00733.x